# Causal vs Non-Causal TabPFN Comparison

In this notebook we reproduce a similar version of the causal vs non-causal comparison done in [TabFlex: Scaling Tabular Learning to Millions with Linear Attention](https://arxiv.org/abs/2506.05584).

In [ ]:
NUM_CLASSES = 2
NUM_FEATURES = 20

NUMBER_OF_TEST_SAMPLES = 100
NUMBER_OF_REPETITIONS = 500

########### PATCHING THE CLASS SAMPLER TO ALWAYS RETURN MAX NUM CLASSES ##############

import tabpfn_prior.priors.flexible_categorical as fc

def class_sampler_f(min_, max_): 
    def s():
        return max_
    return s

fc.class_sampler_f = class_sampler_f

############ END PATCHING THE CLASS SAMPLER ##############

import torch
from tqdm import tqdm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pfns.scripts.tabular_metrics import auc_metric

from pfns.priors.tabpfn_prior_adapter import TabPFNPriorConfig
from pfns.priors.prior import Batch
from pfns.train import compute_losses
from pfns.training_utils import move_style_and_check_shape, move_y_style_and_check_shape
from pfns.utils import torch_nanmean, get_default_device
from pfns.scripts.tabpfn_interface import load_model_workflow



prior = TabPFNPriorConfig(
    prior_type="mlp",
    max_num_classes=NUM_CLASSES,
    max_num_features=NUM_FEATURES,
    flexible=True,
    differentiable=True,
    return_categorical_mask=True,
    nan_handling=True,
)
get_batch = prior.create_get_batch_method()

seqlen_test_list = [200, 400, 800, 900, 1000, 1300, 1600, 2000, 2500, 3000, 4000, 5000]


models_to_compare = {
    "Non-Causal_TabPFN": {
        "base_path": "./../models_diff/tabpfn_prior_config_1_gpu_v4_masking_0/",
    },
    "Causal_TabPFN": {
        "base_path": "./../models_diff/tabpfn_prior_config_1_gpu_v4_masking_0_masking_causal_train_only/",
    }
}

models = {}
configs = {}
for model_name, model_config in models_to_compare.items():
    model, config, _ = load_model_workflow(
        name="checkpoint.pt",
        base_path=model_config["base_path"],
        device=get_default_device(),
    )
    model.eval()
    models[model_name] = model
    configs[model_name] = config


metric_table = {
    name: {"acc": {}, "ce": {}, "roc_auc": {}} for name in models
}

smallest_seqlen = min(seqlen_test_list)
largest_seqlen = max(seqlen_test_list)

for rep in tqdm(range(NUMBER_OF_REPETITIONS), desc="Overall progress"):
    base_batch = get_batch(
        batch_size=1,
        seq_len=largest_seqlen + NUMBER_OF_TEST_SAMPLES,
        num_features=NUM_FEATURES,
        single_eval_pos=largest_seqlen,
        device="cpu",
        n_targets_per_input=1,
    )
    
    # ensure train set and eval set have samples from all classes
    unique_classes_in_train = torch.unique(base_batch.y[:, :smallest_seqlen])
    unique_classes_in_eval = torch.unique(base_batch.y[:, largest_seqlen:largest_seqlen + NUMBER_OF_TEST_SAMPLES])
    while (unique_classes_in_train.numel() < NUM_CLASSES) or (unique_classes_in_eval.numel() < NUM_CLASSES):
        base_batch = get_batch(
            batch_size=1,
            seq_len=largest_seqlen + NUMBER_OF_TEST_SAMPLES,
            num_features=NUM_FEATURES,
            single_eval_pos=largest_seqlen,
            device="cpu",
            n_targets_per_input=1,
        )
        unique_classes_in_train = torch.unique(base_batch.y[:, :smallest_seqlen])
        unique_classes_in_eval = torch.unique(base_batch.y[:, largest_seqlen:largest_seqlen + NUMBER_OF_TEST_SAMPLES])
    
    for model_name, model in models.items():
        config = configs[model_name]

        categorical_inds = None
        if base_batch.categorical_mask is not None:
            mask = base_batch.categorical_mask
            if mask.ndim > 1:
                if not torch.all(mask == mask[0]):
                    raise ValueError("Per-sample categorical masks not supported.")
                mask = mask[0]
            categorical_inds = torch.nonzero(mask, as_tuple=True)[0].tolist()

        with torch.no_grad():
            for seqlen_test in seqlen_test_list:
                x = torch.cat(
                    [
                        base_batch.x[:, :seqlen_test],
                        base_batch.x[:, largest_seqlen : largest_seqlen + NUMBER_OF_TEST_SAMPLES],
                    ],
                    dim=1,
                )
                y = torch.cat(
                    [
                        base_batch.y[:, :seqlen_test],
                        base_batch.y[:, largest_seqlen : largest_seqlen + NUMBER_OF_TEST_SAMPLES],
                    ],
                    dim=1,
                )
                target_y = torch.cat(
                    [
                        base_batch.target_y[:, :seqlen_test],
                        base_batch.target_y[:, largest_seqlen : largest_seqlen + NUMBER_OF_TEST_SAMPLES],
                    ],
                    dim=1,
                )

                test_batch = Batch(
                    x=x,
                    y=y,
                    target_y=target_y,
                    categorical_mask=base_batch.categorical_mask,
                )

                output = model(
                    x=test_batch.x.to(get_default_device()),
                    y=test_batch.y.to(get_default_device()),
                    style=move_style_and_check_shape(test_batch.style, test_batch.x, get_default_device()),
                    y_style=move_y_style_and_check_shape(test_batch.y_style, test_batch.y, get_default_device()),
                    categorical_inds=categorical_inds,
                    only_return_standard_out=True,
                    single_eval_pos=seqlen_test,
                )

                targets = test_batch.target_y[:, seqlen_test:].to(get_default_device())
                losses = compute_losses(output, targets, model.criterion, config.n_targets_per_input)
                loss, _ = torch_nanmean(losses.mean(1), return_nanshare=True)

                pred = output.argmax(dim=-1)
                valid = targets != -100
                if valid.any():
                    accuracy = (pred[valid] == targets[valid]).float().mean().item()
                    
                    probs = torch.softmax(output, dim=-1)
                    auc = auc_metric(targets[valid].cpu(), probs[valid].detach().cpu(), multi_class='ovr')
                    if torch.is_tensor(auc):
                        auc = auc.item()
                else:
                    print("No valid targets for accuracy/auc computation.")
                    accuracy = float("nan")
                    auc = float("nan")

                metric_table[model_name]["acc"].setdefault(seqlen_test, []).append(accuracy)
                metric_table[model_name]["ce"].setdefault(seqlen_test, []).append(float(loss.item()))
                metric_table[model_name]["roc_auc"].setdefault(seqlen_test, []).append(auc)

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.4)
metrics_to_plot = [
    ("acc", "Accuracy"),
    ("ce", "Cross Entropy Loss"),
    ("roc_auc", "ROC AUC")
]

colors = ["#0072B2", "#D55E00"]
markers = ["o", "s"]
linestyles = ["-", "--"]

legend_converter = {}
for i, model_name in enumerate(models):
    legend_converter[model_name] = (markers[i % len(markers)], linestyles[i % len(linestyles)], colors[i % len(colors)])

lw = 3
markersize = 10

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(24, 6))
fig.subplots_adjust(left=0.05, bottom=0.2, right=0.98, top=0.92, wspace=0.25, hspace=0.3)

for idx, (metric_key, metric_name) in enumerate(metrics_to_plot):
    ax = axes[idx]
    
    for model_name in metric_table:
        if metric_key not in metric_table[model_name] or not metric_table[model_name][metric_key]:
            continue
            
        data = metric_table[model_name][metric_key]
        if not data:
            continue
            
        df = pd.DataFrame(data)
        df_transposed = df.transpose()
        mean = df_transposed.mean(axis=1)
        std = df_transposed.std(axis=1)
        x = df.columns.astype(int)

        sns.lineplot(
            x=x,
            y=mean,
            ax=ax,
            label=model_name if idx == 0 else None,
            linestyle=legend_converter[model_name][1],
            color=legend_converter[model_name][2],
            linewidth=lw,
            marker=legend_converter[model_name][0],
            markersize=markersize,
        )

        # ax.fill_between(x, mean - std, mean + std, alpha=0.3, color=legend_converter[model_name][2])

    ax.set_xlabel("Number of Training Samples")
    ax.set_ylabel(metric_name)
    ax.grid(True, which="both", ls="-", alpha=0.2)
    ax.set_title(f"{metric_name} vs Training Samples")

axes[0].legend(fontsize=18)
if axes[1].get_legend(): axes[1].get_legend().remove()
if axes[2].get_legend(): axes[2].get_legend().remove()

plt.show()